In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    print(root, "->", len(files), "files")
    break

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    print(root, "->", len(files), "files")

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

In [4]:
!pip install -q diffusers transformers accelerate peft

In [ ]:
import torch
from diffusers import StableDiffusionPipeline

pipe = StableDiffusionPipeline.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5",
    torch_dtype=torch.float16,
).to("cuda")

In [ ]:
import os
from PIL import Image

os.makedirs("baseline", exist_ok=True)

prompts = [
    "ankara print, large orange sunflower motifs on a navy blue background, bold repeating pattern",
    "ankara print, small white circles and teal geometric shapes on a red background, dense repeating pattern",
    "ankara print, yellow and black fan-shaped shell motifs on a pink background, dense repeating pattern",
    "ankara print, green leaves and purple flowers on a black background, medium-scale repeating pattern",
]

images = []
for i, prompt in enumerate(prompts):
    generator = torch.Generator("cuda").manual_seed(42 + i)
    image = pipe(
        prompt,
        num_inference_steps=30,
        guidance_scale=7.5,
        generator=generator,
    ).images[0]
    image.save(f"baseline/base_{i+1}.png")
    images.append(image)

# put all four side by side so we can see them at once
grid = Image.new("RGB", (512 * 4, 512))
for i, img in enumerate(images):
    grid.paste(img, (512 * i, 0))
grid

In [ ]:
import gc, torch

del pipe
gc.collect()
torch.cuda.empty_cache()

!pip install -q datasets tensorboard

import diffusers
v = diffusers.__version__
print("diffusers version:", v)

!wget -q -O train_lora.py https://raw.githubusercontent.com/huggingface/diffusers/v{v}/examples/text_to_image/train_text_to_image_lora.py
!ls -la train_lora.py

In [ ]:
!pip uninstall -y torchao

In [ ]:
!accelerate launch --num_processes=1 --mixed_precision=fp16 train_lora.py \
  --pretrained_model_name_or_path="stable-diffusion-v1-5/stable-diffusion-v1-5" \
  --train_data_dir="/kaggle/input/datasets/opeyemimuyiwa/ankaragen-dataset" \
  --caption_column="text" \
  --resolution=512 \
  --random_flip \
  --train_batch_size=1 \
  --gradient_accumulation_steps=4 \
  --max_train_steps=1000 \
  --learning_rate=1e-4 \
  --lr_scheduler="cosine" \
  --lr_warmup_steps=0 \
  --rank=4 \
  --mixed_precision="fp16" \
  --checkpointing_steps=250 \
  --seed=42 \
  --output_dir="/kaggle/working/ankara-lora"

In [ ]:
import torch
from diffusers import StableDiffusionPipeline
from PIL import Image
import os

!ls -lh /kaggle/working/ankara-lora/pytorch_lora_weights.safetensors

pipe = StableDiffusionPipeline.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5",
    torch_dtype=torch.float16,
).to("cuda")

pipe.load_lora_weights(
    "/kaggle/working/ankara-lora",
    weight_name="pytorch_lora_weights.safetensors",
)

os.makedirs("after", exist_ok=True)

prompts = [
    "ankara print, large orange sunflower motifs on a navy blue background, bold repeating pattern",
    "ankara print, small white circles and teal geometric shapes on a red background, dense repeating pattern",
    "ankara print, yellow and black fan-shaped shell motifs on a pink background, dense repeating pattern",
    "ankara print, green leaves and purple flowers on a black background, medium-scale repeating pattern",
]

after = []
for i, prompt in enumerate(prompts):
    generator = torch.Generator("cuda").manual_seed(42 + i)
    image = pipe(
        prompt,
        num_inference_steps=30,
        guidance_scale=7.5,
        generator=generator,
    ).images[0]
    image.save(f"after/after_{i+1}.png")
    after.append(image)

# top row = before, bottom row = after
grid = Image.new("RGB", (512 * 4, 512 * 2))
for i in range(4):
    grid.paste(Image.open(f"baseline/base_{i+1}.png"), (512 * i, 0))
    grid.paste(after[i], (512 * i, 512))
grid

In [ ]:
import os, shutil

os.makedirs("/kaggle/working/export", exist_ok=True)

shutil.copy("/kaggle/working/ankara-lora/pytorch_lora_weights.safetensors",
            "/kaggle/working/export/ankaragen_lora_step1000.safetensors")
for step in (500, 750):
    shutil.copy(f"/kaggle/working/ankara-lora/checkpoint-{step}/pytorch_lora_weights.safetensors",
                f"/kaggle/working/export/ankaragen_lora_step{step}.safetensors")

shutil.copytree("/kaggle/working/baseline", "/kaggle/working/export/baseline")
shutil.copytree("/kaggle/working/after", "/kaggle/working/export/after")

shutil.make_archive("/kaggle/working/ankaragen_export", "zip", "/kaggle/working/export")
print(os.path.getsize("/kaggle/working/ankaragen_export.zip") / 1e6, "MB")

In [ ]:
import torch
from PIL import Image

prompts = [
    "ankara print, pink hibiscus flowers and gold leaves on a dark green background, dense repeating pattern",
    "ankara print, red and white zigzag stripes with black dots on a yellow background, bold repeating pattern",
    "ankara print, blue paisley shapes and small orange birds on a cream background, medium-scale repeating pattern",
    "a photo of a cat sitting on a sofa",
]

def generate_row(pipe):
    row = []
    for i, prompt in enumerate(prompts):
        g = torch.Generator("cuda").manual_seed(100 + i)
        img = pipe(prompt, num_inference_steps=30, guidance_scale=7.5, generator=g).images[0]
        row.append(img.resize((384, 384)))
    return row

ckpts = {
    "step500":  ("/kaggle/working/ankara-lora/checkpoint-500",  "pytorch_lora_weights.safetensors"),
    "step750":  ("/kaggle/working/ankara-lora/checkpoint-750",  "pytorch_lora_weights.safetensors"),
    "step1000": ("/kaggle/working/ankara-lora",                 "pytorch_lora_weights.safetensors"),
}

rows = []

# Row 1: base model with no LoRA
pipe.unload_lora_weights()
rows.append(generate_row(pipe))

# Rows 2-4: each checkpoint in turn
for name, (folder, weight) in ckpts.items():
    pipe.load_lora_weights(folder, weight_name=weight)
    rows.append(generate_row(pipe))
    pipe.unload_lora_weights()

grid = Image.new("RGB", (384 * 4, 384 * 4))
for r, row in enumerate(rows):
    for c, img in enumerate(row):
        grid.paste(img, (384 * c, 384 * r))

grid.save("stress_test.png")
grid

In [ ]:
!ls -lh /kaggle/working/ankara-lora/pytorch_lora_weights.safetensors
!ls /kaggle/working

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input/datasets/opeyemimuyiwa/ankaragen-export"):
    print(root, "->", files)

In [ ]:
!pip install -q diffusers transformers accelerate peft
!pip uninstall -y torchao

In [ ]:
import torch
from diffusers import StableDiffusionPipeline

LORA_FOLDER = "/kaggle/input/datasets/opeyemimuyiwa/ankaragen-export"   # change if yours differs

pipe = StableDiffusionPipeline.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5",
    torch_dtype=torch.float16,
).to("cuda")

pipe.load_lora_weights(LORA_FOLDER, weight_name="ankaragen_lora_step1000.safetensors")

image = pipe(
    "ankara print, large orange sunflower motifs on a navy blue background, bold repeating pattern",
    num_inference_steps=30,
    guidance_scale=7.5,
    generator=torch.Generator("cuda").manual_seed(42),
).images[0]
image

In [ ]:
from PIL import Image
import torch

D = "/kaggle/input/datasets/opeyemimuyiwa/ankaragen-export"
prompt = "ankara print, large orange sunflower motifs on a navy blue background, bold repeating pattern"

print("active adapters:", pipe.get_active_adapters())

def gen():
    return pipe(prompt, num_inference_steps=30, guidance_scale=7.5,
                generator=torch.Generator("cuda").manual_seed(42)).images[0]

with_lora = gen()

pipe.disable_lora()
without_lora = gen()
pipe.enable_lora()

grid = Image.new("RGB", (512 * 4, 512))
grid.paste(Image.open(f"{D}/baseline/base_1.png").resize((512, 512)), (0, 0))
grid.paste(Image.open(f"{D}/after/after_1.png").resize((512, 512)), (512, 0))
grid.paste(with_lora.resize((512, 512)), (1024, 0))
grid.paste(without_lora.resize((512, 512)), (1536, 0))
grid

In [ ]:
import torch
from PIL import Image

D = "/kaggle/input/datasets/opeyemimuyiwa/ankaragen-export"

# make sure exactly one LoRA (step 1000) is active
pipe.unload_lora_weights()
pipe.load_lora_weights(D, weight_name="ankaragen_lora_step1000.safetensors")

def set_tiling(pipe, on=True):
    mode = "circular" if on else "zeros"
    for model in (pipe.unet, pipe.vae):
        for m in model.modules():
            if isinstance(m, torch.nn.Conv2d):
                m.padding_mode = mode

def tile_2x2(img, size=256):
    img = img.resize((size, size))
    out = Image.new("RGB", (size * 2, size * 2))
    for x in (0, 1):
        for y in (0, 1):
            out.paste(img, (size * x, size * y))
    return out

test_prompts = [
    "ankara print, large orange sunflower motifs on a navy blue background, bold repeating pattern",
    "ankara print, pink hibiscus flowers and gold leaves on a dark green background, dense repeating pattern",
]

rows = []
for i, prompt in enumerate(test_prompts):
    row = []
    for tiling in (False, True):
        set_tiling(pipe, tiling)
        g = torch.Generator("cuda").manual_seed(7 + i)
        img = pipe(prompt, num_inference_steps=30, guidance_scale=7.5, generator=g).images[0]
        row.append(tile_2x2(img))
    rows.append(row)

set_tiling(pipe, False)

grid = Image.new("RGB", (512 * 2, 512 * 2))
for r, row in enumerate(rows):
    for c, img in enumerate(row):
        grid.paste(img, (512 * c, 512 * r))

grid.save("tiling_test.png")
grid

In [ ]:
import torch
from PIL import Image, ImageDraw

pipe.unload_lora_weights()
pipe.load_lora_weights("/kaggle/input/datasets/opeyemimuyiwa/ankaragen-export",
                       weight_name="ankaragen_lora_step1000.safetensors")

def set_padding(model, mode):
    for m in model.modules():
        if isinstance(m, torch.nn.Conv2d):
            m.padding_mode = mode

def tile_2x2(img, size=192):
    img = img.resize((size, size))
    out = Image.new("RGB", (size * 2, size * 2))
    for x in (0, 1):
        for y in (0, 1):
            out.paste(img, (size * x, size * y))
    return out

prompt = "ankara print, large orange sunflower motifs on a navy blue background, bold repeating pattern"

variants = [
    ("none",      "zeros",    "zeros"),
    ("unet only", "circular", "zeros"),
    ("vae only",  "zeros",    "circular"),
    ("both",      "circular", "circular"),
]

tiles = []
for label, unet_mode, vae_mode in variants:
    set_padding(pipe.unet, unet_mode)
    set_padding(pipe.vae, vae_mode)
    g = torch.Generator("cuda").manual_seed(7)
    img = pipe(prompt, num_inference_steps=30, guidance_scale=7.5, generator=g).images[0]
    t = tile_2x2(img)
    ImageDraw.Draw(t).text((6, 6), label, fill=(255, 255, 255))
    tiles.append(t)

set_padding(pipe.unet, "zeros")
set_padding(pipe.vae, "zeros")

grid = Image.new("RGB", (384 * 4, 384))
for i, t in enumerate(tiles):
    grid.paste(t, (384 * i, 0))
grid.save("tiling_diagnostic.png")
grid

In [ ]:
import torch
from PIL import Image, ImageDraw

def set_padding(model, mode):
    for m in model.modules():
        if isinstance(m, torch.nn.Conv2d):
            m.padding_mode = mode

def half_shift(img):
    w, h = img.size
    out = Image.new("RGB", (w, h))
    # swap quadrants: this moves the original edges to the center
    out.paste(img.crop((w//2, h//2, w, h)), (0, 0))
    out.paste(img.crop((0, h//2, w//2, h)), (w//2, 0))
    out.paste(img.crop((w//2, 0, w, h//2)), (0, h//2))
    out.paste(img.crop((0, 0, w//2, h//2)), (w//2, h//2))
    return out

prompts = [
    "ankara print, large orange sunflower motifs on a navy blue background, bold repeating pattern",
    "ankara print, pink hibiscus flowers and gold leaves on a dark green background, dense repeating pattern",
]

panels = []
for i, prompt in enumerate(prompts):
    for label, mode in (("normal", "zeros"), ("circular", "circular")):
        set_padding(pipe.unet, mode)
        set_padding(pipe.vae, mode)
        g = torch.Generator("cuda").manual_seed(7 + i)
        img = pipe(prompt, num_inference_steps=30, guidance_scale=7.5, generator=g).images[0]
        shifted = half_shift(img).resize((384, 384))
        ImageDraw.Draw(shifted).text((6, 6), label, fill=(255, 255, 255))
        panels.append(shifted)

set_padding(pipe.unet, "zeros")
set_padding(pipe.vae, "zeros")

grid = Image.new("RGB", (384 * 4, 384))
for i, p in enumerate(panels):
    grid.paste(p, (384 * i, 0))
grid.save("seam_check.png")
grid

In [ ]:
!pip install -q gradio

import torch, gradio as gr
from PIL import Image

D = "/kaggle/input/datasets/opeyemimuyiwa/ankaragen-export"

pipe.unload_lora_weights()
pipe.load_lora_weights(D, weight_name="ankaragen_lora_step1000.safetensors")

def set_padding(model, mode):
    for m in model.modules():
        if isinstance(m, torch.nn.Conv2d):
            m.padding_mode = mode

def tile_2x2(img, size=256):
    img = img.resize((size, size))
    out = Image.new("RGB", (size * 2, size * 2))
    for x in (0, 1):
        for y in (0, 1):
            out.paste(img, (size * x, size * y))
    return out

def generate(prompt, tiling, seed, steps, guidance):
    mode = "circular" if tiling else "zeros"
    set_padding(pipe.unet, mode)
    set_padding(pipe.vae, mode)

    prompt = prompt.strip()
    if not prompt.lower().startswith("ankara print"):
        prompt = "ankara print, " + prompt

    g = torch.Generator("cuda").manual_seed(int(seed))
    img = pipe(prompt, num_inference_steps=int(steps),
               guidance_scale=guidance, generator=g).images[0]
    return img, tile_2x2(img)

with gr.Blocks() as demo:
    gr.Markdown("# AnkaraGen\nDescribe an Ankara print and generate it.")
    prompt = gr.Textbox(label="Describe your pattern",
                        value="large pink hibiscus flowers on a dark green background, dense repeating pattern")
    with gr.Row():
        tiling = gr.Checkbox(label="Tileable (circular padding)", value=False)
        seed = gr.Number(label="Seed", value=42, precision=0)
    with gr.Row():
        steps = gr.Slider(10, 50, value=30, step=1, label="Steps")
        guidance = gr.Slider(1, 15, value=7.5, step=0.5, label="Guidance")
    btn = gr.Button("Generate")
    with gr.Row():
        out_img = gr.Image(label="Pattern")
        out_tile = gr.Image(label="2x2 repeat preview")
    btn.click(generate, [prompt, tiling, seed, steps, guidance], [out_img, out_tile])

demo.launch(share=True)

In [ ]:
pipe.safety_checker = None
pipe.requires_safety_checker = False
print("safety checker off")

In [ ]:
import torch
from PIL import Image, ImageDraw

prompt = "ankara print, medium turquoise and coral fish scale shapes with tiny gold dots on a deep purple background, repeating pattern"
seeds = [1, 2, 3, 4, 5, 6, 7, 42]

set_padding(pipe.unet, "circular")
set_padding(pipe.vae, "circular")

size = 256
grid = Image.new("RGB", (size * 4, size * 2))
for i, s in enumerate(seeds):
    g = torch.Generator("cuda").manual_seed(s)
    img = pipe(prompt, num_inference_steps=30, guidance_scale=7.5, generator=g).images[0]
    img = img.resize((size, size))
    ImageDraw.Draw(img).text((6, 6), f"seed {s}", fill=(255, 255, 255))
    grid.paste(img, (size * (i % 4), size * (i // 4)))

set_padding(pipe.unet, "zeros")
set_padding(pipe.vae, "zeros")
grid.save("seed_sweep.png")
grid

In [ ]:
import glob, shutil, os

os.makedirs("/kaggle/working/results_export", exist_ok=True)
for f in glob.glob("/kaggle/working/*.png"):
    shutil.copy(f, "/kaggle/working/results_export")

shutil.make_archive("/kaggle/working/ankaragen_results", "zip", "/kaggle/working/results_export")
print(os.listdir("/kaggle/working/results_export"))